In [1]:
import mysql.connector
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="0987654321",
    database="datamart_rappi"
)

query = """
SELECT 
    fp.tiempo_total_min,
    fp.minutos_retraso,
    fp.dist_asignacion_km,
    fp.tiempo_espera_rest_min,
    fp.cumple_sla,
    dt.hora_dia
FROM fact_pedidos fp
JOIN dim_tiempo dt ON fp.id_tiempo = dt.id_tiempo
LIMIT 5000
"""

df = pd.read_sql(query, conn)
conn.close()
print(f"Datos cargados: {df.shape[0]:,} registros")
print(df.columns.tolist())

Datos cargados: 5,000 registros
['tiempo_total_min', 'minutos_retraso', 'dist_asignacion_km', 'tiempo_espera_rest_min', 'cumple_sla', 'hora_dia']


C:\Users\Admin\AppData\Local\Temp\ipykernel_16588\2662459650.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [3]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="0987654321",
    database="datamart_rappi"
)

query = """
SELECT 
    fp.tiempo_total_min,
    fp.minutos_retraso,
    fp.dist_asignacion_km,
    fp.tiempo_espera_rest_min,
    fp.cumple_sla,
    dt.hora_dia
FROM fact_pedidos fp
JOIN dim_tiempo dt ON fp.id_tiempo = dt.id_tiempo
"""

df = pd.read_sql(query, conn)
conn.close()

print(f"Datos cargados: {df.shape[0]:,} registros")
print(df.columns.tolist())

C:\Users\Admin\AppData\Local\Temp\ipykernel_16588\28148312.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Datos cargados: 10,000 registros
['tiempo_total_min', 'minutos_retraso', 'dist_asignacion_km', 'tiempo_espera_rest_min', 'cumple_sla', 'hora_dia']


In [5]:
print("=" * 60)
print("FASE 1 — MODELO PREDICTIVO")
print("=" * 60)

print("""
JUSTIFICACIÓN DEL ENFOQUE PREDICTIVO:
Se usa un enfoque predictivo porque no basta con describir
lo que pasó (descriptivo), sino que necesitamos ANTICIPAR
si un pedido va a llegar tarde ANTES de asignarlo.

VARIABLE DEPENDIENTE (Y):
→ tiempo_total_min: Tiempo total de entrega en minutos

VARIABLES INDEPENDIENTES (X):
→ X1: dist_asignacion_km   — Distancia del repartidor al restaurante
→ X2: tiempo_espera_rest_min — Tiempo de espera en el restaurante  
→ X3: minutos_retraso       — Minutos de retraso acumulados
""")

# Verificar variables
X = df[["dist_asignacion_km", "tiempo_espera_rest_min", "minutos_retraso"]]
y = df["tiempo_total_min"]

print(f"Variable dependiente Y: {y.name}")
print(f"Variables independientes X: {list(X.columns)}")
print(f"Total registros: {len(df):,}")

FASE 1 — MODELO PREDICTIVO

JUSTIFICACIÓN DEL ENFOQUE PREDICTIVO:
Se usa un enfoque predictivo porque no basta con describir
lo que pasó (descriptivo), sino que necesitamos ANTICIPAR
si un pedido va a llegar tarde ANTES de asignarlo.

VARIABLE DEPENDIENTE (Y):
→ tiempo_total_min: Tiempo total de entrega en minutos

VARIABLES INDEPENDIENTES (X):
→ X1: dist_asignacion_km   — Distancia del repartidor al restaurante
→ X2: tiempo_espera_rest_min — Tiempo de espera en el restaurante  
→ X3: minutos_retraso       — Minutos de retraso acumulados

Variable dependiente Y: tiempo_total_min
Variables independientes X: ['dist_asignacion_km', 'tiempo_espera_rest_min', 'minutos_retraso']
Total registros: 10,000


In [8]:
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# División 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Modelo sklearn para métricas
modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

# R² y MAE
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(((y_test - y_pred) ** 2).mean())

print("=" * 60)
print("RESULTADOS — REGRESIÓN LINEAL")
print("=" * 60)
print(f"R²   (Coef. determinación): {r2:.4f}")
print(f"MAE  (Error promedio):      {mae:.2f} min")
print(f"RMSE (Error cuadrático):    {rmse:.2f} min")

# Coeficientes
coef = pd.DataFrame({
    "Variable": X.columns,
    "Coeficiente": modelo.coef_
})
print("\nCoeficientes del modelo:")
print(coef.to_string(index=False))
print(f"\nIntercepto: {modelo.intercept_:.4f}")

RESULTADOS — REGRESIÓN LINEAL
R²   (Coef. determinación): 0.8300
MAE  (Error promedio):      2.34 min
RMSE (Error cuadrático):    2.92 min

Coeficientes del modelo:
              Variable  Coeficiente
    dist_asignacion_km     4.246854
tiempo_espera_rest_min     1.065949
       minutos_retraso     0.709520

Intercepto: 1.1527


In [9]:
# P-values
X_const = sm.add_constant(X)
modelo_stats = sm.OLS(y, X_const).fit()

print("=" * 60)
print("P-VALUES Y SIGNIFICANCIA ESTADÍSTICA")
print("=" * 60)
print(modelo_stats.summary())

print("\nINTERPRETACIÓN P-VALUES:")
print("Si p-value < 0.05 → variable ES significativa")
print("Si p-value > 0.05 → variable NO es significativa")
print()

for var, pval in zip(X.columns, modelo_stats.pvalues[1:]):
    estado = "Significativa" if pval < 0.05 else "No significativa"
    print(f"{var}: p-value = {pval:.6f} → {estado}")

P-VALUES Y SIGNIFICANCIA ESTADÍSTICA
                            OLS Regression Results                            
Dep. Variable:       tiempo_total_min   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                 1.764e+04
Date:                Tue, 30 Jun 2026   Prob (F-statistic):               0.00
Time:                        23:24:17   Log-Likelihood:                -24730.
No. Observations:               10000   AIC:                         4.947e+04
Df Residuals:                    9996   BIC:                         4.950e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------